# Detecting rooftop available surface for installing PV modules in satellite images using Machine Learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

# Explicit imports — the original `from x import *` chain hid a circular
# import between train.train and hyperparameters.select_param (F-19), and made
# `sigmoid` silently resolve to the NumPy one rather than torch's.
from config import DATA_ROOT, TrainConfig, norm_stats
from utils import get_device, seed_torch, count_parameters
from model.unet import UNet
from loss.loss import iou, accuracy, recall, precision, f1
from loss.losses import ComboLoss, build_loss, compute_pos_weight
from process_data.data_loader import DataLoaderSegmentation
from process_data.import_test import import_and_show
from train.train import training_model, build_optimizer, build_scheduler
from evaluate import evaluate, make_eval_loader
from infer import load_model, predict_mask, predict_large
from plots.plots import plot_train_val, plot_history

%load_ext autoreload
%autoreload 2

In [ ]:
cfg = TrainConfig()          # see config.py for every default
device = get_device()
seed_torch(cfg.seed)

print(f"device: {device}")
print(f"data root: {DATA_ROOT}")

# Loading the Data Set

In [ ]:
# Training crops to cfg.crop (224, a multiple of 32); val/test keep the native
# 250x250 tile and the model boundary pads. Both see identical ground sample
# distance, and nothing is discarded at eval time (F-07).
train_set = DataLoaderSegmentation(
    DATA_ROOT / cfg.train_dir / "images",
    DATA_ROOT / cfg.train_dir / "labels",
    augment=True, crop=cfg.crop, stats_key=cfg.stats_key,
)
val_set = DataLoaderSegmentation(
    DATA_ROOT / cfg.val_dir / "images",
    DATA_ROOT / cfg.val_dir / "labels",
    augment=False, crop=None, stats_key=cfg.stats_key,
)
test_set = DataLoaderSegmentation(
    DATA_ROOT / cfg.test_dir / "images",
    DATA_ROOT / cfg.test_dir / "labels",
    augment=False, crop=None, stats_key=cfg.stats_key,
)

train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, drop_last=False)
val_loader = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers)
test_loader = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers)

print(train_set)
print(val_set)
print(test_set)

# NOTE (F-05): these splits came from a random shuffle of spatially adjacent
# 250x250 crops spaced 62.5 m apart, so train and test leak into each other.
# Regenerate a geographic split before trusting any number from them:
#     python -m process_data.split --root data --out data/splits

In [ ]:
model = UNet(3, 1, False).to(device)
print(f"{count_parameters(model):,} trainable parameters")

# Regular training
This is a simple training loop. We can tune the num_epochs, the learning rate and the parameter of the loss function.

In [ ]:
model = UNet(3, 1, False).to(device)

# F-02: pos_weight, not weight. The 2023 code passed `weight=[4]`, which is a
# uniform per-pixel rescale — mathematically identical to quadrupling the
# learning rate, and useless against class imbalance. build_loss estimates
# pos_weight from the training set and combines BCE with Dice.
loss_function = build_loss(cfg, loader=train_loader, device=device)

# F-09: AdamW @ 3e-4 with a real cosine schedule + warmup. The old run used
# Adam @ 0.01 with StepLR(gamma=1) — a no-op scheduler — so the "decreasing
# learning rate" the report describes never happened.
optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg, steps_per_epoch=len(train_loader))

# F-10: pass val_loader. The 130-epoch run that produced the shipped weights
# omitted it, so there was no validation signal, no early stopping and no
# best-checkpoint selection at all.
history = training_model(
    train_loader,
    loss_function,
    optimizer,
    model,
    num_epochs=cfg.epochs,
    scheduler=scheduler,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
)

print(f"best val IoU {history.best_val_iou:.4f} at epoch {history.best_epoch}")
print(f"artefacts: {history.run_dir}")

In [ ]:
# Validation is now recorded every epoch, so these curves are no longer the
# empty series that ended up in report Figures 15-20 (F-10).
plot_history(history)

## A note on cross-validation

The K-fold cross-validation helpers in `hyperparameters/select_param.py` were
commented out and never ran; that module is now a deprecation shim.

More importantly, K-fold over these tiles would not have helped. The 250×250
crops are windows of one continuous aerial survey spaced 62.5 m apart, so *any*
file-level shuffle — K-fold included — puts a tile's immediate neighbour in the
training fold and the tile itself in the held-out fold. That is the leak
described in F-05.

Use a geographic split instead:

```
python -m process_data.split --root data --out data/splits          # 1 km blocks
python -m process_data.split --inria --root ../dataset/AerialImageDataset/train
python -m process_data.split --holdout-city tyrol-w --root ...      # strictest
```

# Export or import a model

## Export a model 

In [ ]:
# F-18: training_model already writes runs/<timestamp>/{best.pt,last.pt,
# metadata.json,history.json} with the git SHA, full config and val metrics.
# The old cell was `torch.save(..., 'model/' + input('path raise 230') + ".pt")`,
# which needed an interactive prompt and produced four files named
# "path raise NNN.pt" with no record of what they were trained on.
print(f"best checkpoint: {history.run_dir}/best.pt")
print(f"last checkpoint: {history.run_dir}/last.pt")

## Import a model 

In [ ]:
# load_model reads architecture, normalisation and threshold from
# model/manifest.json and calls model.eval() at load time, so no caller can
# forget it (F-03).
CKPT = "model/path raise 130.pt"      # or f"{history.run_dir}/best.pt"

model, entry = load_model(CKPT, device=device)
print(entry)

# Evaluation of the model
We can evaluate the model to have the mean (IoU, Accuracy) on every data set, and print the number of parameters of the Unet.
Mean iou_test, acc_test, recall_test and precision_test

In [ ]:
import pandas as pd

# evaluate() calls model.eval() (F-03), runs under no_grad (F-06), scores empty
# tiles as undefined rather than a free 1.0 (F-08) and guards recall's
# denominator (F-11).
#
# These numbers WILL differ from the 2023 report. That report's figures were
# measured with all four of those bugs live, so they are not valid
# measurements — see potential-fixes.md. Whatever prints below is the project's
# real baseline.
rows = {}
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    rows[name] = evaluate(loader, model, device=device, threshold=cfg.threshold)

cols = ["iou", "f1", "accuracy", "recall", "precision",
        "iou_undefined", "n_images", "positive_pixel_rate"]
pd.DataFrame(rows).T[cols]

# Visualization of the model
With the model we trained or imported, we can display from the test_loader examples of its prediction.

In [ ]:
idx = int(np.random.random() * len(test_loader.dataset))
x, y, original = test_loader.dataset.__getitem__(idx, show_og=True)

from infer import predict_probs

probs = predict_probs(model, x.unsqueeze(0), device=device)[0, 0].cpu().numpy()
ypred = probs > cfg.threshold

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, title in zip(axes, ["Input", "Normalised input", "Expected mask", "Predicted mask"]):
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].imshow(original)
axes[1].imshow(np.clip(np.transpose(x.numpy(), (1, 2, 0)) * 0.25 + 0.5, 0, 1))
axes[2].imshow(y, cmap="gray", vmin=0, vmax=1)
axes[3].imshow(ypred, cmap="gray", vmin=0, vmax=1)
plt.tight_layout()
plt.show()

gt = y.numpy() > 0.5
print(f"IoU       {iou(ypred, gt):.4f}")
print(f"F1        {f1(ypred, gt):.4f}")
print(f"Accuracy  {accuracy(ypred, gt):.4f}")
print("(IoU is nan when the tile genuinely contains no rooftop — F-08)")

In [ ]:
# F-01: this used to be the broken path. It fed the network a diagonally
# transposed, unnormalised 0-255 tensor and then transposed the OUTPUT back, so
# the figures looked plausible while every prediction was corrupt. Every MANIT
# prediction in report Figures 21-23 came through it.
#
# It now delegates to infer.preprocess and keeps native resolution by default —
# the model was trained at ~0.25-0.3 m/px, so resizing changes the effective
# ground sample distance.
mask = import_and_show(model, "Webp.net-resizeimage.png", tta=True)

# Pixels -> area, once you know the ground sample distance of the source image.
GSD_M = 0.25  # metres per pixel; 0.3 for Inria, ~0.274 for z=19 tiles at Bhopal
print(f"detected roof area: {mask.sum() * GSD_M ** 2:,.0f} m^2")